# 19 · The pullback metric: geometry on *learned* manifolds

`omnibias-geometry` now carries one small but powerful primitive: the **pullback
metric** of a chart (immersion) `φ: ℝᵈ → ℝⁿ`,

\[ g = J^\top h\, J, \qquad J = \partial\varphi/\partial x, \]

with `h` the ambient metric (Euclidean by default). The Jacobian `J` is taken by
autodiff, so `g` is *exact* for analytic **and** neural-network charts.

The win: every existing operator (Christoffel, Riemann/Ricci/**scalar
curvature**, Laplace–Beltrami, geodesics) only reads `manifold.metric.g_point`,
so wrapping a chart into a `MetricSpec` makes the whole stack work on learned
manifolds with **no other changes**.

In [ ]:
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from _style import set_style, PRIMARY, ACCENT, GOOD
set_style()

torch.set_default_dtype(torch.float64)

from omnibias.geometry import ChartSpec, ManifoldSpec
from omnibias.geometry.torch import ops as geo

## 1. The standard `S²` embedding induces the round metric

`φ(θ, φ) = (sinθ cosφ, sinθ sinφ, cosθ)` should pull back to the round metric
`diag(1, sin²θ)` and constant scalar curvature `R = 2`.

In [ ]:
def sphere_phi(x):                       # x = (theta, phi)
    th, ph = x[0], x[1]
    return torch.stack([torch.sin(th) * torch.cos(ph),
                        torch.sin(th) * torch.sin(ph),
                        torch.cos(th)])

chart = ChartSpec(phi=sphere_phi, domain_dim=2, ambient_dim=3, name="S2")
manifold = ManifoldSpec("S2", 2, geo.metric_spec_from_chart(chart))

th = torch.linspace(0.2, np.pi - 0.2, 60)
coords = torch.stack([th, torch.full_like(th, 0.7)], dim=-1)
g = geo.pullback_metric(coords, chart)
R = geo.scalar_curvature(coords, manifold)
print(f"scalar curvature: mean = {R.mean():.6f}   (exact = 2)")

fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.6))
ax[0].plot(th, g[:, 0, 0], color=PRIMARY, label="g₀₀ (pullback)")
ax[0].plot(th, g[:, 1, 1], color=ACCENT, label="g₁₁ (pullback)")
ax[0].plot(th, np.sin(th.numpy())**2, "--", color="k", lw=1.2, label="sin²θ (exact)")
ax[0].set_xlabel("θ"); ax[0].set_title("Induced metric components"); ax[0].legend()
ax[1].plot(th, R, color=GOOD); ax[1].axhline(2.0, ls="--", color="k", lw=1.2)
ax[1].set_ylim(1.9, 2.1); ax[1].set_xlabel("θ")
ax[1].set_title("Scalar curvature ≈ 2")
plt.tight_layout()

## 2. A *learned* (neural) chart — exact curvature for free

Now let `φ` be a randomly-initialised MLP `ℝ² → ℝ³`. The pullback metric is
symmetric positive-definite everywhere (a valid Riemannian metric), and omnibias
reads the **exact scalar curvature** of this learned surface.

In [ ]:
torch.manual_seed(0)
net = torch.nn.Sequential(
    torch.nn.Linear(2, 32), torch.nn.Tanh(),
    torch.nn.Linear(32, 32), torch.nn.Tanh(),
    torch.nn.Linear(32, 3),
)
for p in net.parameters():
    p.requires_grad_(False)

nchart = ChartSpec(phi=net, domain_dim=2, ambient_dim=3, name="mlp")
nman = ManifoldSpec("learned", 2, geo.metric_spec_from_chart(nchart))

u = torch.linspace(-1, 1, 28)
U, V = torch.meshgrid(u, u, indexing="ij")
grid = torch.stack([U.reshape(-1), V.reshape(-1)], dim=-1)

gm = geo.pullback_metric(grid, nchart)
min_eig = torch.linalg.eigvalsh(gm).min()
print(f"min metric eigenvalue over grid: {min_eig:.3e}   (>0 ⇒ valid Riemannian metric)")

Rg = geo.scalar_curvature(grid, nman).reshape(28, 28)
fig, ax = plt.subplots(figsize=(5.4, 4.3))
im = ax.pcolormesh(U.numpy(), V.numpy(), Rg.numpy(), shading="auto", cmap="coolwarm")
fig.colorbar(im, label="scalar curvature R(z)")
ax.set_xlabel("z₁"); ax.set_ylabel("z₂")
ax.set_title("Exact curvature of a *learned* manifold")
plt.tight_layout()

## Takeaway

A chart + one autodiff Jacobian turns omnibias's exact differential geometry into
a toolkit for **learned** manifolds. This single primitive is what the manifold
learning, shape analysis, and equivariance notebooks build on.